# 4000 senaryo: MC, Wick ve yeni analitik varyans

Mevcut `environments_4000.csv` ve reponun `prepare_teacher_bank` akışı kullanılır.
Her bankada **aynı 32 W × 512 Z** için üç sonuç hesaplanır:

| Sütun | Hesap |
|---|---|
| `varEmp` | Python kanal üretecinden MC güç varyansı |
| `sigma2Wick` | Mevcut `evaluate_gamma_batch` Wick hesabı |
| `sigma2Hop` | Yeni koşullu varyans formülü; tam alıcılar arası kovaryans |

$Y=\|G\,\operatorname{diag}(\gamma)H w\|^2$; MC varyansı population normalizasyonunu kullanır.
Yeni formül bağımsız proper kompleks Gaussian **hoplar** için tamdır; kaskadı Gaussian saymaz.
Sonlu ışın/XPR kanalına ne kadar uyduğunu burada üretilen MC ile ölçüyoruz.
MC örnekleri analitik girdilere girmez. Bu notebook eğitim yapmaz; MATLAB dosyası gerekmez.

Varsayılan kapsam dosyadaki **4000 bankanın tamamı**, bütün W/Z adayları ve banka başına
**64.000 kanal çifti**dir. Dosyanın train/validation/test etiketleri sonuçlarda korunur.


## 1. Drive ve GitHub kodu

Colab'da GPU seçin. Bu deneyin kodu ayrı klasöre klonlanır; mevcut çalışma kopyanız korunur.
Yerel Jupyter'de `REPO_DIR` mevcut repo köküne ayarlanabilir.


In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = "https://github.com/OsmanMertYilmaz/Phy_Informed_MIMO_RIS_Env.git"
REF = "codex/analytic-native-mc-python"
IN_COLAB = "google.colab" in sys.modules
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT = Path("/content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env")
    REPO_DIR = Path("/content/ris_analytic_mc_python")
else:
    REPO_DIR = Path.cwd()
    if REPO_DIR.name == "notebooks":
        REPO_DIR = REPO_DIR.parent
    PROJECT = REPO_DIR / "data"

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", REF, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
if not (REPO_DIR / "src/ris_env/variance_comparison.py").exists():
    raise RuntimeError(f"Deney kodu bulunamadı: {REPO_DIR}. {REF} dalını kullanın.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "matplotlib"], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))
print("Kod:", REPO_DIR)


## 2. Aynı 4000 senaryo ve deney ayarları

`ENV_CSV` mevcut eğitim üretiminde kullanılan dosya olmalıdır; otomatik yeni geometri oluşturulmaz.
`BANK_IDS=None`, `W_INDICES=None`, `Z_INDICES=None` tüm kapsamı korur. Küçük bir deneme için
bu listelere açıkça indeks yazabilirsiniz; W/Z indeksleri sıfır tabanlıdır.

`PARITY=True` çift hassasiyet kullanır. Yeni tam kovaryans hesabı Wick'ten daha pahalıdır;
`ANALYTIC_Z_CHUNK` yalnızca çalışma belleğini/hızı ayarlar, aday sayısını azaltmaz.
MC örnek sayısını, hassasiyeti veya aday listesini değiştirdiğinizde yeni bir `RUN_TAG` seçin.


In [ ]:
import gc, json, time
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from ris_env.teacher_pipeline import prepare_teacher_bank
from ris_env.variance_comparison import compare_prepared_bank, summarize_comparisons

ENV_CSV = PROJECT / "environments" / "environments_4000.csv"
RUN_TAG = "mc_wick_new_analytic_4000_v1"
OUTPUT = PROJECT / "analytic_variance_comparison" / RUN_TAG

BANK_IDS = None                 # None: CSV'deki 4000 bankanın tamamı
W_INDICES = None                # None: mevcut 32 W
Z_INDICES = None                # None: mevcut 512 Z
N_MC = 64_000
MC_CHUNK = 256
MC_CHECKPOINT_SAMPLES = 4096
W_CHUNK = 4
Z_CHUNK = 64
ANALYTIC_Z_CHUNK = 4
PARITY = True                  # complex128 / float64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not ENV_CSV.is_file():
    raise FileNotFoundError(f"Mevcut 4000 senaryo dosyasını ENV_CSV ile gösterin: {ENV_CSV}")
environments = pd.read_csv(ENV_CSV)
if len(environments) != 4000 or environments.bankID.duplicated().any():
    raise ValueError("4000 benzersiz bankID içeren ortam CSV'si bekleniyor.")
if BANK_IDS is not None:
    missing = set(BANK_IDS) - set(environments.bankID)
    if missing:
        raise ValueError(f"Bilinmeyen bankID: {missing}")
    environments = environments[environments.bankID.isin(BANK_IDS)].copy()
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Cihaz:", DEVICE, "| Hassasiyet:", "float64" if PARITY else "float32")
print("Bankalar:", len(environments), "| MC/banka:", N_MC)
print("W:", "32 (tümü)" if W_INDICES is None else W_INDICES)
print("Z:", "512 (tümü)" if Z_INDICES is None else Z_INDICES)
print("Çıktı:", OUTPUT)


## 3. MC kanallarını üret ve üç varyansı hesapla

Kanallar her MC parçasında bir kez üretilir ve bütün seçili W/Z adaylarında paylaşılır.
Mevcut MC motoru, mevcut Wick API'si ve yeni analitik modül çağrılır.
MC RNG durumu ve toplamları kaydedilir; analitik hesap her W tamamlandığında kaydedilir.
Aynı ayarlarla hücreyi yeniden çalıştırmak kaldığı yerden devam eder.

`comparison.parquet` her banka için tüm karşılaştırmaları içerir.
`bank_summary.csv` ve `plot_sample.csv` her tamamlanan bankadan sonra güncellenir.
Grafik örneklemesi yalnızca çizim içindir; karşılaştırma metrikleri bütün seçili adaylardan hesaplanır.
Aynı çıktı klasöründe eşzamanlı iki oturum çalıştırmayın.


In [ ]:
summaries = []
plot_samples = []
started = time.perf_counter()
for number, (_, row) in enumerate(environments.iterrows(), 1):
    bank_id = int(row.bankID)
    print(f"\n[{number}/{len(environments)}] bankID={bank_id}: ortam ve adaylar hazırlanıyor", flush=True)
    prepared = prepare_teacher_bank(row, k_w=32, device=DEVICE, parity=PARITY, z_chunk=Z_CHUNK)
    folder = OUTPUT / f"bank_{bank_id:06d}"
    result = compare_prepared_bank(
        prepared, folder, n_mc=N_MC, mc_chunk=MC_CHUNK,
        w_chunk=W_CHUNK, z_chunk=Z_CHUNK, analytic_z_chunk=ANALYTIC_Z_CHUNK,
        checkpoint_samples=MC_CHECKPOINT_SAMPLES,
        w_indices=W_INDICES, z_indices=Z_INDICES,
    )
    summary = {"bankID": bank_id, **summarize_comparisons(result)}
    for name in ["scenario_BR", "scenario_RU", "nRIS", "splitID", "split"]:
        if name in row.index:
            summary[name] = row[name]
    summaries.append(summary)
    plot_samples.append(result.sample(min(64, len(result)), random_state=20260911))
    bank_summary = pd.DataFrame(summaries)
    summary_tmp = OUTPUT / "bank_summary.csv.tmp"
    bank_summary.to_csv(summary_tmp, index=False)
    summary_tmp.replace(OUTPUT / "bank_summary.csv")
    plot_tmp = OUTPUT / "plot_sample.csv.tmp"
    pd.concat(plot_samples, ignore_index=True).to_csv(plot_tmp, index=False)
    plot_tmp.replace(OUTPUT / "plot_sample.csv")
    print(f"Wick medyan mutlak hata: %{summary['wickMedianAbsErrorPct']:.3f} | "
          f"Yeni analitik: %{summary['hopMedianAbsErrorPct']:.3f} | "
          f"Geçersiz: {summary['invalidRows']} | Geçen: {(time.perf_counter()-started)/60:.1f} dk")
    del result, prepared
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("Tamamlandı:", OUTPUT)
display(bank_summary)


## 4. Karşılaştırma özeti

Aşağıdaki tablo **banka bazında** medyan mutlak yüzde hataları karşılaştırır.
Aynı bankadaki W/Z satırları bağımsız örnekler sayılmaz.
`analyticValid=False` satırlar, negatif/sanal artık veya Wick tutarlılık kontrolü nedeniyle
ayrı sayılır; ham yeni analitik değerler kırpılmadan Parquet dosyasında saklanır.
Sıfır MC varyansında yüzde hata tanımsızdır ve özetten çıkarılan satır sayısına dahildir.


In [ ]:
bank_summary = pd.read_csv(OUTPUT / "bank_summary.csv")
metric_columns = ["wickMedianAbsErrorPct", "hopMedianAbsErrorPct"]
display(bank_summary[metric_columns].describe(percentiles=[.5, .9, .95]).T)
print("Geçersiz / yüzde hata hesaplanamayan satırlar:", int(bank_summary.invalidRows.sum()))
valid_banks = bank_summary.validRows > 0
print("Yeni analitiğin banka medyan hatasını düşürdüğü banka oranı: %",
      round(100 * (bank_summary.loc[valid_banks, "hopMedianAbsErrorPct"] <
                   bank_summary.loc[valid_banks, "wickMedianAbsErrorPct"]).mean(), 2))
for grouping in [["scenario_BR", "scenario_RU"], ["nRIS"]]:
    if all(name in bank_summary for name in grouping):
        print("Banka medyanlarının medyanı:", grouping)
        display(bank_summary.groupby(grouping)[metric_columns].median())


In [ ]:
import matplotlib.pyplot as plt
plot_data = pd.read_csv(OUTPUT / "plot_sample.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, column, title in zip(axes, ["sigma2Wick", "sigma2Hop"], ["Wick", "Yeni analitik"]):
    valid = plot_data.analyticValid & (plot_data.varEmp > 0) & (plot_data[column] > 0)
    d = plot_data.loc[valid]
    if len(d):
        ax.scatter(d.varEmp, d[column], s=5, alpha=.2)
        lo = min(d.varEmp.min(), d[column].min()); hi = max(d.varEmp.max(), d[column].max())
        ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set(xscale="log", yscale="log", xlabel="MC güç varyansı", ylabel="Analitik güç varyansı", title=title)
    ax.grid(True, which="both", alpha=.2)
fig.tight_layout()
fig.savefig(OUTPUT / "mc_wick_new_analytic.png", dpi=180)
plt.show()


## 5. İstenilen bankayı ve MC yakınsamasını incele

Yakınsama tablosu aynı kanal akışının artan örnek sayılarındaki varyansıdır.
Bu noktalar bağımsız tekrarlar değildir; bu hücre yeni kanal üretmez.


In [ ]:
BANK_TO_INSPECT = int(bank_summary.bankID.iloc[0])
W_TO_INSPECT = 0 if W_INDICES is None else int(W_INDICES[0])
Z_TO_INSPECT = 0 if Z_INDICES is None else int(Z_INDICES[0])
folder = OUTPUT / f"bank_{BANK_TO_INSPECT:06d}"
full = pd.read_parquet(folder / "comparison.parquet")
display(full[["bankID", "wCandidate", "zCandidate", "varEmp", "sigma2Wick", "sigma2Hop",
              "wickAbsErrorPct", "hopAbsErrorPct", "analyticValid"]].head(20))
manifest = json.loads((folder / "manifest.json").read_text())
ki = manifest["w_indices"].index(W_TO_INSPECT)
zi = manifest["z_indices"].index(Z_TO_INSPECT)
selected = full[(full.wCandidate == W_TO_INSPECT) & (full.zCandidate == Z_TO_INSPECT)].iloc[0]
curve = []
for path in sorted(folder.glob("mc_at_*.npz")):
    with np.load(path, allow_pickle=False) as data:
        curve.append({"N_MC": int(path.stem.split("_")[-1]), "varEmp": float(data["varEmp"][ki, zi]),
                      "sigma2Wick": selected.sigma2Wick, "sigma2Hop": selected.sigma2Hop})
convergence = pd.DataFrame(curve)
display(convergence)
if len(convergence):
    ax = convergence.plot(x="N_MC", y=["varEmp", "sigma2Wick", "sigma2Hop"], figsize=(8, 4), grid=True)
    ax.set_ylabel("Güç varyansı")
    plt.show()
